# FABenv Phase 2 — PPO 训练可视化

本 Notebook 运行 Phase 2 SAS PPO 训练，并可视化训练过程中最关键的奖励、损失、完成度与终止原因。

默认配置：
- 训练模式：`small` 固定小规模问题实例
- Episode 数：`NUM_EPISODES = 50`
- 可视化布局：2×3 等高网格 + 单独的 step-level 奖励分解图

## 实时看曲线：推荐使用 TensorBoard

这个 Notebook 仍然适合“训练完成后做静态分析”。如果你想**边训练边看 reward/loss 曲线**，推荐在终端训练时打开 TensorBoard 日志：

```bash
python FAB_RL/FABenv/train_phase2_sas_ppo.py --mode small --episodes 200 --tensorboard-logdir FAB_RL/FABenv/runs/phase2_small
```

然后另开一个终端启动 TensorBoard：

```bash
tensorboard --logdir FAB_RL/FABenv/runs
```

TensorBoard 会打印一个本地网址，通常是：

```text
http://localhost:6006
```

在浏览器打开后，可以看到这些实时更新的曲线：

- `train/episode_reward`
- `train/completed_lots`
- `train/steps`
- `train/wait_steps`
- `train/failed_actions`
- `loss/policy_loss`
- `loss/value_loss`
- `loss/entropy`

注意：曲线是**每完成一个 episode 新增一个点**，不是每个 step 都新增一个点。

## 1. 导入依赖与运行 PPO 训练

In [ ]:
import os
import sys
from collections import Counter

fab_path = os.path.abspath('.')
if fab_path not in sys.path:
    sys.path.insert(0, fab_path)

import matplotlib.pyplot as plt
import numpy as np

from phase2_ppo_buffer import Phase2RolloutBuffer
from train_phase2_sas_ppo import build_training_components

%matplotlib inline
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 110

NUM_EPISODES = 50
MOVING_AVERAGE_WINDOW = 5

components = build_training_components()
trainer = components['trainer']
driver = components['driver']
config = trainer.config

history = []
last_buffer = None

print(f'开始训练: {NUM_EPISODES} episodes, small mode')
for episode in range(NUM_EPISODES):
    driver.reset_episode()
    buffer = Phase2RolloutBuffer(
        gamma=config.gamma,
        gae_lambda=config.gae_lambda,
    )
    summary = trainer.collect_episode(driver, buffer, stochastic=True)

    if buffer.steps:
        buffer.finish_episode(last_value=0.0)
        stats = trainer.update_policy(buffer)
    else:
        stats = {'policy_loss': 0.0, 'value_loss': 0.0, 'entropy': 0.0}

    history.append({'episode': episode, **summary, **stats})
    last_buffer = buffer

    if episode == 0 or (episode + 1) % 10 == 0:
        print(
            f"Episode {episode + 1:3d}/{NUM_EPISODES} | "
            f"reward={summary['episode_reward']:.2f} | "
            f"completed_lots={summary['completed_lots']} | "
            f"steps={summary['steps']} | "
            f"policy_loss={stats['policy_loss']:.4f} | "
            f"value_loss={stats['value_loss']:.4f}"
        )

print('训练完成。')

## 2. Episode 级训练曲线（2×3 网格）

该图组展示训练过程中的主要 episode-level 指标：奖励、PPO 损失、熵、完成 Lot 数与终止原因。

In [ ]:
def moving_average(values, window=MOVING_AVERAGE_WINDOW):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode='valid')


def moving_average_x(values, window=MOVING_AVERAGE_WINDOW):
    values = np.asarray(values)
    if len(values) < window:
        return values
    return values[window - 1:]


def plot_curve(ax, episodes, values, title, ylabel, color):
    ax.plot(episodes, values, alpha=0.35, color=color, linewidth=0.9, label='raw')
    if len(values) >= MOVING_AVERAGE_WINDOW:
        ax.plot(
            moving_average_x(episodes),
            moving_average(values),
            color=color,
            linewidth=2.0,
            label=f'MA({MOVING_AVERAGE_WINDOW})',
        )
    ax.set_title(title)
    ax.set_xlabel('Episode')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)


episodes = [row['episode'] for row in history]
episode_rewards = [row['episode_reward'] for row in history]
policy_losses = [row['policy_loss'] for row in history]
value_losses = [row['value_loss'] for row in history]
entropies = [row['entropy'] for row in history]
completed_lots = [row['completed_lots'] for row in history]
steps = [row['steps'] for row in history]
termination_reasons = [row['termination_reason'] for row in history]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Phase 2 SAS PPO Training Curves', fontsize=15, fontweight='bold')

plot_curve(axes[0, 0], episodes, episode_rewards, 'Episode Reward', 'Reward', 'C0')
plot_curve(axes[0, 1], episodes, policy_losses, 'Policy Loss', 'Loss', 'C1')
plot_curve(axes[0, 2], episodes, value_losses, 'Value Loss', 'Loss', 'C2')
plot_curve(axes[1, 0], episodes, entropies, 'Entropy', 'Entropy', 'C3')

ax = axes[1, 1]
ax.plot(episodes, completed_lots, color='C4', linewidth=1.8, label='Completed Lots')
ax.set_title('Completed Lots / Steps')
ax.set_xlabel('Episode')
ax.set_ylabel('Completed Lots', color='C4')
ax.tick_params(axis='y', labelcolor='C4')
ax.grid(True, alpha=0.3)
ax_right = ax.twinx()
ax_right.plot(episodes, steps, color='C5', linewidth=1.5, linestyle='--', label='Steps')
ax_right.set_ylabel('Steps', color='C5')
ax_right.tick_params(axis='y', labelcolor='C5')
lines_left, labels_left = ax.get_legend_handles_labels()
lines_right, labels_right = ax_right.get_legend_handles_labels()
ax.legend(lines_left + lines_right, labels_left + labels_right, fontsize=8, loc='best')

ax = axes[1, 2]
reason_counts = Counter(termination_reasons)
labels = list(reason_counts.keys())
sizes = list(reason_counts.values())
colors = plt.cm.Set2(np.linspace(0, 1, max(len(labels), 1)))
ax.pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct='%1.0f%%',
    startangle=90,
    textprops={'fontsize': 8},
)
ax.set_title('Termination Reason')

plt.tight_layout()
plt.show()

## 3. 单 Episode 奖励分量分解

下图从最后一个 episode 的 rollout buffer 中读取每个 step 的 `StepInfo`，展示各奖励分量随决策步推进的变化。

In [ ]:
if last_buffer is None or not last_buffer.steps:
    print('无可用 buffer 数据，跳过奖励分量分解。')
else:
    reward_components = [
        ('reward_execute', 'execute'),
        ('reward_wait', 'wait'),
        ('reward_tardy', 'tardy'),
        ('reward_qtime', 'qtime'),
        ('reward_priority', 'priority'),
        ('reward_progress', 'progress'),
        ('reward_shape', 'shape'),
        ('reward_terminal', 'terminal'),
    ]

    x = np.arange(len(last_buffer.steps))
    component_values = {}
    for field_name, label in reward_components:
        values = np.array([getattr(step.info, field_name, 0.0) for step in last_buffer.steps], dtype=float)
        if not np.allclose(values, 0.0):
            component_values[label] = values

    if not component_values:
        print('所有奖励分量均为 0，跳过奖励分量分解。')
    else:
        labels = list(component_values.keys())
        data = np.vstack([component_values[label] for label in labels])

        fig, ax = plt.subplots(figsize=(14, 5))
        ax.stackplot(
            x,
            data,
            labels=labels,
            colors=plt.cm.Set2(np.linspace(0, 1, len(labels))),
            alpha=0.85,
        )
        ax.plot(
            x,
            [step.reward for step in last_buffer.steps],
            color='black',
            linewidth=1.2,
            label='total step reward',
        )
        ax.axhline(0.0, color='black', linewidth=0.6, alpha=0.7)
        ax.set_title('Reward Component Decomposition — Last Episode')
        ax.set_xlabel('Step Index')
        ax.set_ylabel('Reward')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper left', fontsize=8, ncol=4)
        plt.tight_layout()
        plt.show()

## 4. 训练总结

In [ ]:
reward_array = np.asarray(episode_rewards, dtype=float)
best_idx = int(np.argmax(reward_array))

print(f'Episodes:           {NUM_EPISODES}')
print(f'Reward min/max:     {reward_array.min():.2f} / {reward_array.max():.2f}')
print(f'Reward mean/last:   {reward_array.mean():.2f} / {reward_array[-1]:.2f}')
print(f'Best episode:       {best_idx} (reward={reward_array[best_idx]:.2f})')
print(f'Average steps:      {np.mean(steps):.1f}')
print(f'Average lots done:  {np.mean(completed_lots):.1f}')
print('Termination reasons:')
for reason, count in Counter(termination_reasons).most_common():
    print(f'  - {reason}: {count} ({count / NUM_EPISODES * 100:.0f}%)')